# Reliability Patterns — Hands-On

**LLM Engineering · Domain 8 · Roadmap Week 22**

Offline notebook: retries with exponential backoff and jitter, fallback simulation, circuit breaker state machine, rate-limit queue, and dead-letter queue.

## 0. Setup

In [ ]:
%pip install -q numpy
import random, numpy as np
rng = np.random.RandomState(48)
print("ok")

## 1. Retry schedule without sleeping

In [ ]:
def schedule(base=.1, factor=2, cap=2, attempts=5, seed=1):
    r = random.Random(seed); out = []
    for i in range(attempts):
        raw = min(cap, base*(factor**i)); out.append(round(raw + r.uniform(0, raw*.25), 3))
    return out
print(schedule())

## 2. Retry simulator

In [ ]:
TRANSIENT = {"timeout", "429", "500"}
def call_with_policy(outcomes, max_attempts=4):
    delays = schedule(attempts=max_attempts)
    elapsed = 0
    for i, outcome in enumerate(outcomes[:max_attempts]):
        if outcome == "ok": return {"status":"ok", "attempts":i+1, "elapsed":round(elapsed,3)}
        if outcome not in TRANSIENT: return {"status":"fail_fast", "reason":outcome, "attempts":i+1}
        elapsed += delays[i]
    return {"status":"failed", "attempts":min(len(outcomes), max_attempts), "elapsed":round(elapsed,3)}
print(call_with_policy(["500", "timeout", "ok"]))
print(call_with_policy(["schema_error", "ok"]))

## 3. Fallback model/provider

In [ ]:
def provider(name, outcome):
    if outcome == "ok": return {"provider": name, "answer": "answer", "degraded": False}
    raise RuntimeError(outcome)
def with_fallback(primary_outcome, fallback_outcome):
    try: return provider("primary", primary_outcome)
    except RuntimeError as e:
        try:
            ans = provider("fallback", fallback_outcome); ans["degraded"] = True; return ans
        except RuntimeError: return {"provider": None, "answer": "Please try again later", "degraded": True}
print(with_fallback("timeout", "ok"))

## 4. Circuit breaker state machine

In [ ]:
class CircuitBreaker:
    def __init__(self, threshold=2, reset_after=3):
        self.threshold=threshold; self.reset_after=reset_after; self.failures=0; self.state="closed"; self.opened_at=None
    def allow(self, tick):
        if self.state == "open" and tick - self.opened_at >= self.reset_after:
            self.state = "half_open"; return True
        return self.state != "open"
    def record(self, ok, tick):
        if ok: self.failures=0; self.state="closed"; self.opened_at=None
        else:
            self.failures += 1
            if self.failures >= self.threshold: self.state="open"; self.opened_at=tick
cb = CircuitBreaker(); outcomes = [False, False, True, True, False, False, False, True]
for tick, ok in enumerate(outcomes):
    allowed = cb.allow(tick); print(tick, "before", cb.state, "allow", allowed)
    if allowed: cb.record(ok, tick)

## 5. Rate-limit token bucket

In [ ]:
capacity, refill, tokens = 5, 2, 5
requests = [3,4,2,1,5]
for minute, demand in enumerate(requests):
    tokens = min(capacity, tokens + refill)
    accepted = min(tokens, demand); tokens -= accepted
    rejected = demand - accepted
    print(minute, "accepted", accepted, "rejected", rejected, "tokens_left", tokens)

## 6. Idempotency and dead-letter queue

In [ ]:
processed, dlq = {}, []
def handle(job):
    key = job["idempotency_key"]
    if key in processed: return processed[key]
    if job["attempts"] > 2:
        dlq.append(job); return {"status":"dead_lettered"}
    result = {"status":"sent", "ticket_id": "T-" + key[-3:]}
    processed[key] = result; return result
job = {"idempotency_key":"email-123", "attempts":1}
print(handle(job)); print(handle(job))
print(handle({"idempotency_key":"email-999", "attempts":3}), dlq)

## 7. Exercises
1. Add `Retry-After` handling to the schedule.
2. Make the circuit breaker track success rate in half-open state.
3. Add per-tenant token buckets.
4. Add a fallback that returns retrieval snippets instead of generated prose.

## Links
- Literature note: `02 Literature Notes/LLM Engineering/Reliability Patterns`
- Snippets: `04 Code Snippets/LLM/Retry Backoff With Jitter Simulator`, `.../Circuit Breaker State Machine`